# Common Data Cleaning - RT_IOT2022 Dataset

## Objective

This notebook performs common data cleaning on the RT_IOT2022 dataset. The cleaned dataset will serve as the shared source for all three ML tracks (Regression, Classification, and Clustering). Common cleaning includes loading, inspecting, and addressing data quality issues that affect all tracks equally. Track-specific preprocessing (scaling, encoding, splitting) will be performed separately by each track.

## Load Dataset

We load the raw RT_IOT2022 dataset from the data directory. The file is in CSV format but has no extension.

In [1]:
import pandas as pd
import numpy as np

DATA_PATH = "../data/RT_IOT2022"

df = pd.read_csv(DATA_PATH)
df.head()

,Unnamed: 0,id.orig_p,id.resp_p,proto,service,flow_duration,fwd_pkts_tot,bwd_pkts_tot,fwd_data_pkts_tot,bwd_data_pkts_tot,...,active.std,idle.min,idle.max,idle.tot,idle.avg,idle.std,fwd_init_window_size,bwd_init_window_size,fwd_last_window_size,Attack_type
0,0,38667,1883,tcp,mqtt,32.011598,9,5,3,3,...,0.0,2.972918e+07,2.972918e+07,2.972918e+07,2.972918e+07,0.0,64240,26847,502,MQTT_Publish
1,1,51143,1883,tcp,mqtt,31.883584,9,5,3,3,...,0.0,2.985528e+07,2.985528e+07,2.985528e+07,2.985528e+07,0.0,64240,26847,502,MQTT_Publish
2,2,44761,1883,tcp,mqtt,32.124053,9,5,3,3,...,0.0,2.984215e+07,2.984215e+07,2.984215e+07,2.984215e+07,0.0,64240,26847,502,MQTT_Publish
3,3,60893,1883,tcp,mqtt,31.961063,9,5,3,3,...,0.0,2.991377e+07,2.991377e+07,2.991377e+07,2.991377e+07,0.0,64240,26847,502,MQTT_Publish
4,4,51087,1883,tcp,mqtt,31.902362,9,5,3,3,...,0.0,2.981470e+07,2.981470e+07,2.981470e+07,2.981470e+07,0.0,64240,26847,502,MQTT_Publish


## Initial Dataset Audit

We examine the dataset shape, column names, and data types to understand the structure and identify any immediate issues.

In [2]:
print("Dataset shape:", df.shape)
print("\nNumber of columns:", len(df.columns))
print("\nColumn names:")
print(df.columns.tolist())

Dataset shape: (123117, 85)

Number of columns: 85

Column names:
['Unnamed: 0', 'id.orig_p', 'id.resp_p', 'proto', 'service', 'flow_duration', 'fwd_pkts_tot', 'bwd_pkts_tot', 'fwd_data_pkts_tot', 'bwd_data_pkts_tot', 'fwd_pkts_per_sec', 'bwd_pkts_per_sec', 'flow_pkts_per_sec', 'down_up_ratio', 'fwd_header_size_tot', 'fwd_header_size_min', 'fwd_header_size_max', 'bwd_header_size_tot', 'bwd_header_size_min', 'bwd_header_size_max', 'flow_FIN_flag_count', 'flow_SYN_flag_count', 'flow_RST_flag_count', 'fwd_PSH_flag_count', 'bwd_PSH_flag_count', 'flow_ACK_flag_count', 'fwd_URG_flag_count', 'bwd_URG_flag_count', 'flow_CWR_flag_count', 'flow_ECE_flag_count', 'fwd_pkts_payload.min', 'fwd_pkts_payload.max', 'fwd_pkts_payload.tot', 'fwd_pkts_payload.avg', 'fwd_pkts_payload.std', 'bwd_pkts_payload.min', 'bwd_pkts_payload.max', 'bwd_pkts_payload.tot', 'bwd_pkts_payload.avg', 'bwd_pkts_payload.std', 'flow_pkts_payload.min', 'flow_pkts_payload.max', 'flow_pkts_payload.tot', 'flow_pkts_payload.avg', 

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123117 entries, 0 to 123116
Data columns (total 85 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   Unnamed: 0                123117 non-null  int64  
 1   id.orig_p                 123117 non-null  int64  
 2   id.resp_p                 123117 non-null  int64  
 3   proto                     123117 non-null  object 
 4   service                   123117 non-null  object 
 5   flow_duration             123117 non-null  float64
 6   fwd_pkts_tot              123117 non-null  int64  
 7   bwd_pkts_tot              123117 non-null  int64  
 8   fwd_data_pkts_tot         123117 non-null  int64  
 9   bwd_data_pkts_tot         123117 non-null  int64  
 10  fwd_pkts_per_sec          123117 non-null  float64
 11  bwd_pkts_per_sec          123117 non-null  float64
 12  flow_pkts_per_sec         123117 non-null  float64
 13  down_up_ratio             123117 non-null  f

In [4]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Unnamed: 0,123117.0,NaN,NaN,NaN,37035.089248,30459.106367,0.0,6059.0,33100.0,63879.0,94658.0
id.orig_p,123117.0,NaN,NaN,NaN,34639.258738,19070.620354,0.0,17702.0,37221.0,50971.0,65535.0
id.resp_p,123117.0,NaN,NaN,NaN,1014.305092,5256.371994,0.0,21.0,21.0,21.0,65389.0
proto,123117,3,tcp,110427,NaN,NaN,NaN,NaN,NaN,NaN,NaN
service,123117,10,-,102861,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
idle.std,123117.0,NaN,NaN,NaN,45501.831692,1091361.401715,0.0,0.0,0.0,0.0,120802870.628073
fwd_init_window_size,123117.0,NaN,NaN,NaN,6118.905123,18716.313861,0.0,64.0,64.0,64.0,65535.0
bwd_init_window_size,123117.0,NaN,NaN,NaN,2739.776018,10018.848534,0.0,0.0,0.0,0.0,65535.0
fwd_last_window_size,123117.0,NaN,NaN,NaN,751.647514,6310.183843,0.0,64.0,64.0,64.0,65535.0


## Missing Value Check

We check every column for missing values to determine whether imputation is required. The cleaning strategy will be based on the observed missing-value pattern.

In [5]:
missing_values = df.isnull().sum()
missing_values_sorted = missing_values.sort_values(ascending=False)
print("Missing values per column:")
print(missing_values_sorted[missing_values_sorted > 0] if missing_values_sorted.sum() > 0 else "No missing values found")

Missing values per column:
No missing values found


## Duplicate Check

We check for duplicate rows to ensure data quality. Duplicate records can bias model training and should be addressed.

In [6]:
duplicate_count = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count}")
print(f"Percentage of duplicate rows: {duplicate_count / len(df) * 100:.2f}%")

Number of duplicate rows: 0
Percentage of duplicate rows: 0.00%


## Data Type Check

We verify that each column has an appropriate data type. Incorrect data types can indicate data quality issues or require correction.

In [7]:
print("Data types per column:")
print(df.dtypes)

Data types per column:
Unnamed: 0                int64
id.orig_p                 int64
id.resp_p                 int64
proto                    object
service                  object
                         ...   
idle.std                float64
fwd_init_window_size      int64
bwd_init_window_size      int64
fwd_last_window_size      int64
Attack_type              object
Length: 85, dtype: object


## Constant / Near-Constant Feature Check

We identify columns with zero or very low variance. Constant features provide no predictive information and should be removed. Near-constant features may also be problematic for some models.

In [8]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
constant_cols = []
near_constant_cols = []

for col in numeric_cols:
    unique_count = df[col].nunique()
    if unique_count == 1:
        constant_cols.append(col)
    elif unique_count <= 2:
        near_constant_cols.append((col, unique_count))

print(f"Constant columns (zero variance): {len(constant_cols)}")
if constant_cols:
    print(constant_cols)

print(f"\nNear-constant columns (≤2 unique values): {len(near_constant_cols)}")
if near_constant_cols:
    for col, count in near_constant_cols:
        print(f"  {col}: {count} unique values")

Constant columns (zero variance): 1
['bwd_URG_flag_count']

Near-constant columns (≤2 unique values): 1
  fwd_URG_flag_count: 2 unique values


## Index-like / Unnecessary Column Check

We identify columns that appear to be index-like or otherwise unnecessary for modeling. These include sequential row identifiers, columns with all unique values, or columns that serve no predictive purpose.

In [9]:
print("Checking for index-like columns:")

if 'Unnamed: 0' in df.columns:
    unnamed_unique = df['Unnamed: 0'].nunique()
    unnamed_range = df['Unnamed: 0'].max() - df['Unnamed: 0'].min() + 1
    print(f"Unnamed: 0 - Unique values: {unnamed_unique}, Range: {unnamed_range}")
    if unnamed_unique == len(df) and unnamed_range == len(df):
        print("  -> Appears to be a row index")

all_unique_cols = []
for col in df.columns:
    if df[col].nunique() == len(df):
        all_unique_cols.append(col)

print(f"\nColumns with all unique values: {len(all_unique_cols)}")
if all_unique_cols:
    print(all_unique_cols)

Checking for index-like columns:
Unnamed: 0 - Unique values: 94659, Range: 94659



Columns with all unique values: 0


## Outlier Check

We examine numerical features for extreme values. Network traffic data naturally contains unusual flows, especially attack traffic. We document outliers without automatically removing them, as extreme values may represent legitimate attack observations.

In [10]:
numeric_cols = df.select_dtypes(include=[np.number]).columns

print("Statistical summary for numerical columns:")
print(df[numeric_cols].describe().T)

Statistical summary for numerical columns:


                         count          mean           std  min           25%  \
Unnamed: 0            123117.0  3.703509e+04  3.045911e+04  0.0   6059.000000   
id.orig_p             123117.0  3.463926e+04  1.907062e+04  0.0  17702.000000   
id.resp_p             123117.0  1.014305e+03  5.256372e+03  0.0     21.000000   
flow_duration         123117.0  3.809566e+00  1.300054e+02  0.0      0.000001   
fwd_pkts_tot          123117.0  2.268826e+00  2.233656e+01  0.0      1.000000   
...                        ...           ...           ...  ...           ...   
idle.avg              123117.0  1.664985e+06  9.007064e+06  0.0      0.000000   
idle.std              123117.0  4.550183e+04  1.091361e+06  0.0      0.000000   
fwd_init_window_size  123117.0  6.118905e+03  1.871631e+04  0.0     64.000000   
bwd_init_window_size  123117.0  2.739776e+03  1.001885e+04  0.0      0.000000   
fwd_last_window_size  123117.0  7.516475e+02  6.310184e+03  0.0     64.000000   

                           

In [11]:
print("Checking for extreme values in key columns:")

key_columns = ['flow_duration', 'fwd_pkts_tot', 'bwd_pkts_tot']
for col in key_columns:
    if col in df.columns:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
        print(f"\n{col}:")
        print(f"  IQR range: [{lower_bound:.2f}, {upper_bound:.2f}]")
        print(f"  Outliers: {len(outliers)} ({len(outliers)/len(df)*100:.2f}%)")
        print(f"  Min: {df[col].min():.2f}, Max: {df[col].max():.2f}")

Checking for extreme values in key columns:

flow_duration:
  IQR range: [-0.00, 0.00]
  Outliers: 19171 (15.57%)
  Min: 0.00, Max: 21728.34

fwd_pkts_tot:
  IQR range: [1.00, 1.00]
  Outliers: 17662 (14.35%)
  Min: 0.00, Max: 4345.00

bwd_pkts_tot:
  IQR range: [1.00, 1.00]
  Outliers: 32364 (26.29%)
  Min: 0.00, Max: 10112.00


## Cleaning Decisions

Based on the data inspection, we make the following cleaning decisions:

1. **Index-like column**: `Unnamed: 0` appears to be a row index (0 to 123116) with no predictive value. It will be removed.

2. **Constant columns**: `bwd_URG_flag_count` has zero variance (only one unique value) and provides no predictive information. It will be removed.

3. **Missing values**: The dataset has no missing values, so no imputation is required.

4. **Duplicates**: After removing the index column, 5,195 duplicate rows were revealed. These are exact duplicates across all feature columns and should be removed to prevent biasing model training.

5. **Outliers**: Extreme values in network traffic data may represent legitimate attack observations. We will document them but preserve them unless there is a clear data quality issue. The outlier check showed 15-26% of observations fall outside IQR bounds for key columns, which is expected for network traffic data.

6. **Target columns**: Both `Attack_type` (classification target) and `flow_duration` (regression target) will be preserved.

7. **No track-specific preprocessing**: Scaling, encoding, and train/test split will be handled separately by each track.

## Apply Cleaning

We apply the cleaning decisions to create the cleaned dataset.

In [12]:
df_cleaned = df.copy()

print("Original shape:", df_cleaned.shape)

if 'Unnamed: 0' in df_cleaned.columns:
    df_cleaned = df_cleaned.drop(columns=['Unnamed: 0'])
    print("Removed: Unnamed: 0 (index column)")

numeric_cols_cleaned = df_cleaned.select_dtypes(include=[np.number]).columns
constant_cols_cleaned = []
for col in numeric_cols_cleaned:
    if df_cleaned[col].nunique() == 1:
        constant_cols_cleaned.append(col)

if constant_cols_cleaned:
    df_cleaned = df_cleaned.drop(columns=constant_cols_cleaned)
    print(f"Removed constant columns: {constant_cols_cleaned}")
else:
    print("No constant columns found")

duplicate_count_before = df_cleaned.duplicated().sum()
print(f"\nDuplicate rows before removal: {duplicate_count_before}")

if duplicate_count_before > 0:
    df_cleaned = df_cleaned.drop_duplicates()
    print(f"Removed {duplicate_count_before} duplicate rows")

print("Cleaned shape:", df_cleaned.shape)

Original shape: (123117, 85)
Removed: Unnamed: 0 (index column)
Removed constant columns: ['bwd_URG_flag_count']



Duplicate rows before removal: 5195


Removed 5195 duplicate rows
Cleaned shape: (117922, 83)


## Post-Cleaning Verification

We verify the cleaned dataset to ensure all cleaning steps were applied correctly and the data is ready for use by all tracks.

In [13]:
print("=== Cleaned Dataset Verification ===")
print(f"Final shape: {df_cleaned.shape}")
print(f"Final column count: {len(df_cleaned.columns)}")

missing_after = df_cleaned.isnull().sum().sum()
print(f"Missing values: {missing_after}")

duplicates_after = df_cleaned.duplicated().sum()
print(f"Duplicate rows: {duplicates_after}")

print(f"\nData types summary:")
print(df_cleaned.dtypes.value_counts())

=== Cleaned Dataset Verification ===
Final shape: (117922, 83)
Final column count: 83
Missing values: 0


Duplicate rows: 0

Data types summary:
float64    56
int64      24
object      3
Name: count, dtype: int64


In [14]:
print("Target columns preserved:")
print(f"  Attack_type (classification): {df_cleaned['Attack_type'].nunique()} unique values")
print(f"  flow_duration (regression): min={df_cleaned['flow_duration'].min():.2f}, max={df_cleaned['flow_duration'].max():.2f}")

print(f"\nAttack_type distribution:")
print(df_cleaned['Attack_type'].value_counts())

Target columns preserved:
  Attack_type (classification): 12 unique values
  flow_duration (regression): min=0.00, max=21728.34

Attack_type distribution:
Attack_type
DOS_SYN_Hping                 90089
Thing_Speak                    7654
ARP_poisioning                 7625
MQTT_Publish                   4142
NMAP_UDP_SCAN                  2584
NMAP_XMAS_TREE_SCAN            2010
NMAP_OS_DETECTION              2000
NMAP_TCP_scan                  1002
DDOS_Slowloris                  533
Wipro_bulb                      219
Metasploit_Brute_Force_SSH       36
NMAP_FIN_SCAN                    28
Name: count, dtype: int64


In [15]:
print("Numerical summary of cleaned dataset:")
print(df_cleaned.describe().T)

Numerical summary of cleaned dataset:
                         count          mean           std  min           25%  \
id.orig_p             117922.0  3.494909e+04  1.896605e+04  0.0  18352.250000   
id.resp_p             117922.0  1.050812e+03  5.364537e+03  0.0     21.000000   
flow_duration         117922.0  3.808474e+00  1.270642e+02  0.0      0.000001   
fwd_pkts_tot          117922.0  2.290209e+00  2.211145e+01  0.0      1.000000   
bwd_pkts_tot          117922.0  1.942776e+00  3.372767e+01  0.0      1.000000   
...                        ...           ...           ...  ...           ...   
idle.avg              117922.0  1.736783e+06  9.194955e+06  0.0      0.000000   
idle.std              117922.0  4.721069e+04  1.114041e+06  0.0      0.000000   
fwd_init_window_size  117922.0  6.248730e+03  1.889360e+04  0.0     64.000000   
bwd_init_window_size  117922.0  2.816081e+03  1.016753e+04  0.0      0.000000   
fwd_last_window_size  117922.0  7.401732e+02  6.242324e+03  0.0     64.

## Save Cleaned Dataset

We save the cleaned dataset to the data directory. This cleaned dataset will be the common source for all three ML tracks.

In [16]:
cleaned_path = "../data/RT_IOT2022_cleaned.csv"
df_cleaned.to_csv(cleaned_path, index=False)
print(f"Cleaned dataset saved to: {cleaned_path}")

Cleaned dataset saved to: ../data/RT_IOT2022_cleaned.csv


## Verification: Reload Saved Dataset

We reload the saved CSV to verify it can be loaded successfully and matches the in-memory cleaned dataframe.

In [17]:
df_reloaded = pd.read_csv(cleaned_path)
print(f"Reloaded dataset shape: {df_reloaded.shape}")
print(f"Original cleaned shape: {df_cleaned.shape}")

if df_reloaded.shape == df_cleaned.shape:
    print("\nShapes match - verification successful")
else:
    print("\nShape mismatch - verification failed")

print(f"\nReloaded columns match: {list(df_reloaded.columns) == list(df_cleaned.columns)}")

Reloaded dataset shape: (117922, 83)
Original cleaned shape: (117922, 83)

Shapes match - verification successful

Reloaded columns match: True
